In [1]:
import cv2
import time
import argparse
import numpy as np
import torch
import torch.nn as nn
import scipy.signal
from scipy.signal import butter, filtfilt
import collections


# ----------------------------
# EfficientPhys model
# ----------------------------

class Attention_mask(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        xsum = torch.sum(x, dim=2, keepdim=True)
        xsum = torch.sum(xsum, dim=3, keepdim=True)
        xshape = tuple(x.size())
        return x / (xsum + 1e-8) * xshape[2] * xshape[3] * 0.5


class TSM(nn.Module):
    def __init__(self, n_segment=10, fold_div=3):
        super().__init__()
        self.n_segment = n_segment
        self.fold_div = fold_div

    def forward(self, x):
        nt, c, h, w = x.size()
        n_batch = nt // self.n_segment
        x = x.view(n_batch, self.n_segment, c, h, w)

        fold = c // self.fold_div
        out = torch.zeros_like(x)

        out[:, :-1, :fold] = x[:, 1:, :fold]
        out[:, 1:, fold:2 * fold] = x[:, :-1, fold:2 * fold]
        out[:, :, 2 * fold:] = x[:, :, 2 * fold:]

        return out.view(nt, c, h, w)


class EfficientPhys(nn.Module):
    def __init__(
        self,
        in_channels=3,
        nb_filters1=32,
        nb_filters2=64,
        kernel_size=3,
        dropout_rate1=0.25,
        dropout_rate2=0.5,
        pool_size=(2, 2),
        nb_dense=128,
        frame_depth=20,
        img_size=36,
        channel="raw",
    ):
        super().__init__()

        self.in_channels = in_channels
        self.kernel_size = kernel_size
        self.dropout_rate1 = dropout_rate1
        self.dropout_rate2 = dropout_rate2
        self.pool_size = pool_size
        self.nb_filters1 = nb_filters1
        self.nb_filters2 = nb_filters2
        self.nb_dense = nb_dense

        self.TSM_1 = TSM(n_segment=frame_depth)
        self.TSM_2 = TSM(n_segment=frame_depth)
        self.TSM_3 = TSM(n_segment=frame_depth)
        self.TSM_4 = TSM(n_segment=frame_depth)

        self.motion_conv1 = nn.Conv2d(in_channels, nb_filters1, kernel_size=kernel_size, padding=1)
        self.motion_conv2 = nn.Conv2d(nb_filters1, nb_filters1, kernel_size=kernel_size)
        self.motion_conv3 = nn.Conv2d(nb_filters1, nb_filters2, kernel_size=kernel_size, padding=1)
        self.motion_conv4 = nn.Conv2d(nb_filters2, nb_filters2, kernel_size=kernel_size)

        self.apperance_att_conv1 = nn.Conv2d(nb_filters1, 1, kernel_size=1)
        self.attn_mask_1 = Attention_mask()
        self.apperance_att_conv2 = nn.Conv2d(nb_filters2, 1, kernel_size=1)
        self.attn_mask_2 = Attention_mask()

        self.avg_pooling_1 = nn.AvgPool2d(pool_size)
        self.avg_pooling_3 = nn.AvgPool2d(pool_size)

        self.dropout_1 = nn.Dropout(dropout_rate1)
        self.dropout_3 = nn.Dropout(dropout_rate1)
        self.dropout_4 = nn.Dropout(dropout_rate2)

        if img_size == 36:
            self.final_dense_1 = nn.Linear(3136, nb_dense)
        elif img_size == 72:
            self.final_dense_1 = nn.Linear(16384, nb_dense)
        elif img_size == 96:
            self.final_dense_1 = nn.Linear(30976, nb_dense)
        else:
            raise ValueError("Unsupported image size")

        self.final_dense_2 = nn.Linear(nb_dense, 1)
        self.batch_norm = nn.BatchNorm2d(3)
        self.channel = channel

    def forward(self, inputs, params=None):
        inputs = torch.diff(inputs, dim=0)
        inputs = self.batch_norm(inputs)

        network_input = self.TSM_1(inputs)

        d1 = torch.tanh(self.motion_conv1(network_input))
        d1 = self.TSM_2(d1)
        d2 = torch.tanh(self.motion_conv2(d1))

        g1 = torch.sigmoid(self.apperance_att_conv1(d2))
        g1 = self.attn_mask_1(g1)
        gated1 = d2 * g1

        d3 = self.avg_pooling_1(gated1)
        d4 = self.dropout_1(d3)

        d4 = self.TSM_3(d4)
        d5 = torch.tanh(self.motion_conv3(d4))
        d5 = self.TSM_4(d5)
        d6 = torch.tanh(self.motion_conv4(d5))

        g2 = torch.sigmoid(self.apperance_att_conv2(d6))
        g2 = self.attn_mask_2(g2)
        gated2 = d6 * g2

        d7 = self.avg_pooling_3(gated2)
        d8 = self.dropout_3(d7)

        d9 = d8.view(d8.size(0), -1)
        d10 = torch.tanh(self.final_dense_1(d9))
        d11 = self.dropout_4(d10)

        return self.final_dense_2(d11)


# ----------------------------
# Signal processing
# ----------------------------

def bandpass(signal, fs, low, high, order=2):
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, signal)

def fft_rate(signal, fs, low_hz, high_hz):
    signal = np.asarray(signal).astype(np.float64)
    signal = signal - np.mean(signal)

    if len(signal) < fs * 4:
        return None

    freqs, power = scipy.signal.periodogram(signal, fs=fs, detrend=False)

    mask = (freqs >= low_hz) & (freqs <= high_hz)
    if not np.any(mask):
        return None

    peak_freq = freqs[mask][np.argmax(power[mask])]
    return float(peak_freq * 60.0)

def estimate_hr(bvp, fs):
    try:
        filtered = bandpass(bvp, fs, 0.75, 2.5)
        return fft_rate(filtered, fs, 0.75, 2.5)
    except Exception:
        return None

def estimate_rr_from_bvp(bvp, fs):
    """
    Approximate respiration rate from low-frequency modulation.
    This is weaker than using a model trained directly for respiration.
    """
    try:
        filtered = bandpass(bvp, fs, 0.10, 0.50)
        return fft_rate(filtered, fs, 0.10, 0.50)
    except Exception:
        return None

# ----------------------------
# Video preprocessing
# ----------------------------

def crop_center_face_or_frame(frame):
    """
    Simplified ROI. For better accuracy, replace this with face detection.
    """
    h, w, _ = frame.shape
    size = int(min(h, w) * 0.65)

    cx, cy = w // 2, h // 2
    x1 = max(cx - size // 2, 0)
    y1 = max(cy - size // 2, 0)
    x2 = min(cx + size // 2, w)
    y2 = min(cy + size // 2, h)

    return frame[y1:y2, x1:x2], (x1,y1,x2,y2)

def preprocess_frame(frame, img_size=36):
    roi, (x1,y1,x2,y2) = crop_center_face_or_frame(frame)
    roi = cv2.resize(roi, (img_size, img_size))
    roi = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)

    roi = roi.astype(np.float32) / 255.0
    roi = np.transpose(roi, (2, 0, 1))

    return roi, (x1,y1,x2,y2)

def load_model(weight_path, device, frame_depth=20, img_size=36):
    model = EfficientPhys(frame_depth=frame_depth, img_size=img_size)
    checkpoint = torch.load(weight_path, map_location=device)

    if isinstance(checkpoint, dict):
        if "state_dict" in checkpoint:
            checkpoint = checkpoint["state_dict"]
        elif "model_state_dict" in checkpoint:
            checkpoint = checkpoint["model_state_dict"]

    cleaned = {}
    for k, v in checkpoint.items():
        k = k.replace("module.", "")
        cleaned[k] = v

    model.load_state_dict(cleaned, strict=False)
    model.to(device)
    model.eval()

    return model

In [2]:
def main():
    WEIGHTS_PATH = "/Users/saptarshimallikthakur/Downloads/iBVP_EfficientPhys.pth"
    SOURCE = 0
    IMG_SIZE = 72
    FRAME_DEPTH = 20
    WINDOW_SEC = 10
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    device = torch.device(DEVICE)

    model = load_model(
        WEIGHTS_PATH,
        device=device,
        frame_depth=FRAME_DEPTH,
        img_size=IMG_SIZE
    )

    cap = cv2.VideoCapture(SOURCE)

    if not cap.isOpened():
        raise RuntimeError(f"Could not open source: {SOURCE}")

    frame_buffer = []
    bvp_buffer = []

    # Initialize variables with safe defaults
    fs = 30.0  
    hr = None
    rr = None
    last_infer_time = 0

    print(f"Running on: {device}")
    print(f"Using IMG_SIZE: {IMG_SIZE}")
    print(f"Using model frame depth: {FRAME_DEPTH}")

    # Track timestamps of the last 30 frames to compute a rolling FPS
    timestamps = collections.deque(maxlen=30)

    while True:
        ret, frame = cap.read()

        # 1. Check if frame was read successfully immediately
        if not ret:
            break

        # 2. Record timestamp for FPS tracking
        timestamps.append(time.time())

        # 3. Calculate actual FPS once we have at least 2 frames
        if len(timestamps) > 1:
            fs = len(timestamps) / (timestamps[-1] - timestamps[0])
        else:
            fs = 30.0  # Fallback for the very first frame

        max_bvp_len = int(WINDOW_SEC * fs)
        display = frame.copy()

        # 4. Preprocess and append to frame buffer
        processed, (x1, y1, x2, y2) = preprocess_frame(frame, IMG_SIZE)
        frame_buffer.append(processed)

        # Keep buffer size strictly to FRAME_DEPTH + 1
        if len(frame_buffer) > FRAME_DEPTH + 1:
            frame_buffer.pop(0)

        # Once we have enough frames, run the model
        if len(frame_buffer) == FRAME_DEPTH + 1:
            clip = np.stack(frame_buffer, axis=0)
            clip = torch.tensor(clip, dtype=torch.float32, device=device)

            with torch.no_grad():
                pred = model(clip).detach().cpu().numpy().flatten()

            # Append only the newest BVP point
            bvp_buffer.append(pred[-1]) 

            # Keep BVP buffer at max length
            if len(bvp_buffer) > max_bvp_len:
                bvp_buffer.pop(0)

            now = time.time()

            # Run FFT when we have at least 6 seconds of BVP data
            if len(bvp_buffer) >= int(fs * 6) and now - last_infer_time > 1.0:
                bvp_np = np.array(bvp_buffer)

                hr = estimate_hr(bvp_np, fs)
                rr = estimate_rr_from_bvp(bvp_np, fs)

                last_infer_time = now

        # 5. UI Overlays
        hr_text = "HR: estimating..." if hr is None else f"HR: {hr:.1f} bpm"
        rr_text = "RR: estimating..." if rr is None else f"RR: {rr:.1f} rpm"

        cv2.rectangle(display, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(display, hr_text, (30, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

        cv2.putText(display, rr_text, (30, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 200, 0), 2)

        cv2.imshow("EfficientPhys HR / RR", display)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q") or key == 27:
            break

    print(f"Final Estimated FPS: {fs:.2f}")
    cap.release()
    cv2.destroyAllWindows()

main()

Running on: cpu
Using IMG_SIZE: 72
Using model frame depth: 20
Final Estimated FPS: 10.35
